In [1]:
import os
import numpy as np
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt

from diffusion_lightning import (
    DiffusionModel, FieldDataModule, 
    grab, get_mag, get_abs_mag, get_chi2, get_UL, mag
)

# Create directories
for folder in ['data', 'figures', 'models']:
    os.makedirs(folder, exist_ok=True)

print(f"Using GPU: {torch.cuda.get_device_name(2)}")

Using GPU: NVIDIA GeForce RTX 5090


## 1. Configuration

In [2]:
L = 128          
k = 0.5  
l = 0.022       
sigma = 25.0


n_epochs = 250
batch_size = 256
lr = 1e-3

num_steps = 1000
# num_samples = 1024

data_path = f"data/cfgs_k={k}_l={l}_128^2_t=10.jld2"

## 2. Prepare Data

In [4]:
data_module = FieldDataModule(
    data_path=data_path,
    batch_size=batch_size,
    normalize=True
)
data_module.setup()

print(f"Training samples: {len(data_module.train_data)}")

Training samples: 10240


In [5]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D/diffusion_L128_k0.5_l0.022_128^2_t=100-epoch=999.ckpt"

In [6]:
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:2"))


## 4. Generate Samples

In [ ]:
# MAALA Sampling (Metropolis-Adjusted Annealed Langevin Algorithm)
# 中间步骤使用标准 EM 方法，只在最后几步添加 MH 校正

num_samples_mala = 128

samples_mala, acceptance_rate = model.sample_mala(
    num_samples=num_samples_mala,
    num_steps=500,
    k=k,
    l=l,
    mh_last_steps=10,
    alpha=0.01,
)

print(f"\nMAALA: {num_samples_mala} samples, acceptance rate: {acceptance_rate:.4f}")

MAALA:  63%|██████▎   | 316/500 [00:33<00:20,  9.12it/s]

In [ ]:
# 可视化 MAALA 样本
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples_mala[i, 0].cpu(), cmap='RdBu')
    ax.axis('off')
plt.suptitle(f'MAALA Samples (acc rate: {acceptance_rate:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# Compare MAALA samples with standard DM samples
samples_mala_re = grab(samples_mala[:, 0, :, :])
cfgs_dm_mala = np.array(samples_mala_re, dtype=np.float64, copy=True)
cfgs_dm_mala2 = np.array(np.concatenate([samples_mala_re, -samples_mala_re], axis=0), dtype=np.float64, copy=True)

# This comparison requires running the standard DM sampling first (Cell 8)
# and the cumulant analysis functions (Cell 18)
print("MAALA samples shape:", samples_mala.shape)
print("MAALA samples range: [{:.4f}, {:.4f}]".format(samples_mala_re.min(), samples_mala_re.max()))

In [ ]:
# Compare cumulants: Standard DM vs MAALA DM
# Note: Run Cell 18 first to define lattice_bootstrap_cumulants()

# Compute cumulants for MALA samples
cumulants_mala, errs_mala = lattice_bootstrap_cumulants(cfgs_dm_mala, 8, n_boot=10, seed=3)
cumulants_mala2, errs_mala2 = lattice_bootstrap_cumulants(cfgs_dm_mala2, 8, n_boot=10, seed=3)

print("\nComparison: Standard DM vs MAALA DM")
print("Type       | κ_2         | κ_4           | κ_6           | κ_8")
print("-" * 70)
print("Data       ", end="|")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
    print(f"{c:.4f}±{e:.4f}", end=" |")
print()
print("DM (std)   ", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
    print(f"{c:.4f}±{e:.4f}", end=" |")
print()
print("DM (MAALA) ", end="|")
for c, e in zip(cumulants_mala2[1::2], errs_mala2[1::2]):
    print(f"{c:.4f}±{e:.4f}", end=" |")

In [10]:
num_samples_generate = 512   
sample_batch_size = 64

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)

KeyboardInterrupt: 

In [ ]:
import numpy as np
from scipy.stats import moment
from scipy.special import binom
import h5py
def norm(x, dtype=np.float64, copy=True):
    """Normalize array to [-1, 1] using min/max."""
    x = np.array(x, dtype=dtype, copy=copy)
    xmin = x.min()
    xmax = x.max()
    scale = xmax - xmin
    if scale == 0:
        x.fill(dtype(0))
        return x
    x -= xmin
    x /= scale
    x -= dtype(0.5)
    x *= dtype(2.0)
    return x


def cumulants_from_moments(moments):
    """Cumulants from moments using the recursion in Eq. (1.221).

    `moments` has shape (order, ...). The output has the same shape.
    """
    m = np.asarray(moments, dtype=np.float64)
    order = m.shape[0]
    kappa = np.empty_like(m, dtype=np.float64)
    kappa[0] = m[0]
    for n in range(2, order + 1):
        val = m[n - 1].copy()
        for i in range(1, n):
            val -= binom(n - 1, i - 1) * kappa[i - 1] * m[n - i - 1]
        kappa[n - 1] = val
    return kappa


def _prepare_site_data(data, dtype=np.float64, lattice_axes=(-2, -1)):
    """Return (x_flat, site_shape) with x_flat shaped (n_samples, n_sites).

    We compute local (single-site) cumulants κ_n(x). Any non-lattice, non-sample
    axes (e.g. a channel axis of length 1) are averaged out first.
    """
    x = np.asarray(data)
    if x.ndim == 4 and x.shape[1] == 1:
        x = x[:, 0]
    x = np.asarray(x, dtype=dtype)

    nd = x.ndim
    lat_axes = tuple(ax if ax >= 0 else nd + ax for ax in lattice_axes)
    other_axes = tuple(ax for ax in range(1, nd) if ax not in lat_axes)
    if other_axes:
        x = x.mean(axis=other_axes, dtype=np.float64)

    site_shape = x.shape[1:]
    x_flat = x.reshape(x.shape[0], -1)
    return x_flat, site_shape


def lattice_bootstrap_cumulants(
    data,
    order,
    axis=0,
    n_boot=1000,
    seed=None,
    dtype=np.float64,
    lattice_axes=(-2, -1),
    n_bins=1000,
):
    """Bootstrap estimate of local cumulants κ_n.

    Paper definition (local cumulants):
      1) compute κ_n(x) at each lattice site x from the ensemble,
      2) average over sites: κ_n = (1/V) Σ_x κ_n(x).

    Errors are estimated by bootstrapping over *bins* of configurations.
    """
    rng = np.random.default_rng(seed)

    x_flat, site_shape = _prepare_site_data(data, dtype=dtype, lattice_axes=lattice_axes)
    n_samples, n_sites = x_flat.shape

    n_bins = int(min(n_bins, n_samples))
    bin_size = n_samples // n_bins
    if bin_size < 1:
        raise ValueError("n_bins is too large for the number of samples")

    n_used = bin_size * n_bins
    if n_used != n_samples:
        x_flat = x_flat[:n_used]
        n_samples = n_used

    # Per-bin, per-site moments: shape (n_bins, order, n_sites)
    per_bin_m = np.empty((n_bins, order, n_sites), dtype=np.float64)
    for b in range(n_bins):
        xb = x_flat[b * bin_size : (b + 1) * bin_size].astype(np.float64, copy=False)
        x_pow = xb.copy()
        for k in range(order):
            per_bin_m[b, k] = x_pow.mean(axis=0, dtype=np.float64)
            if k != order - 1:
                x_pow *= xb

    boot_kappa = np.empty((n_boot, order), dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n_bins, size=n_bins, dtype=np.int32)
        m = per_bin_m[idx].mean(axis=0, dtype=np.float64)  # (order, n_sites)
        kappa = cumulants_from_moments(m)  # (order, n_sites)
        boot_kappa[b] = kappa.mean(axis=1)

    cumulants_means = boot_kappa.mean(axis=0)
    boot_errors = boot_kappa.std(axis=0)
    return cumulants_means, boot_errors

In [ ]:
# cfgs_original = norm(np.load("data/field_theory/cfgs_L32_k0.5_l0.022_10k.npy", mmap_mode="r"))

In [ ]:
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=10, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=10, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=10, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=10, seed=1)

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D/diffusion_L128_k0.5_l0.022_128^2_t=100-epoch=999.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:3"))
num_samples_generate = 512   
sample_batch_size = 256

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
    last_step=300,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=999-tau=0.6.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
data_module.setup()

In [ ]:
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=999.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D/diffusion_L128_k0.5_l0.022_128^2_t=100-epoch=499.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:0"))
num_samples_generate = 512   
sample_batch_size = 128

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=10, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=10, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=10, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=10, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=499.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D/diffusion_L128_k0.5_l0.022_128^2_t=100-epoch=49.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:0"))
num_samples_generate = 512   
sample_batch_size = 128

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=10, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=10, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=10, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=10, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=49.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D/diffusion_L128_k0.5_l0.022_128^2_t=100-epoch=249.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:0"))
num_samples_generate = 512   
sample_batch_size = 128

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=10, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=10, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=10, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=10, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=249.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D-10/diffusion_L128_k0.5_l0.022_128^2_t=10-epoch=00-step=0005.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:3"))
num_samples_generate = 512   
sample_batch_size = 256

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=00-step=0005.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
modeldir="/home/tyy/DM/DMasSQ-main/models/2D-10/diffusion_L128_k0.5_l0.022_128^2_t=10-epoch=03.ckpt"
model = DiffusionModel.load_from_checkpoint(modeldir).to(torch.device("cuda:3"))
num_samples_generate = 512   
sample_batch_size = 256

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,
    sample_batch_size=sample_batch_size,
    show_progress=True,
)
samples_re = grab(samples[:, 0, :, :])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=10, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=10, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=10, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=10, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=03.npy", data_module.renorm(cfgs_dm3).transpose(1,2,0))

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
print("dm k1,k3,k5,k7 =", cumulants_cfgs_dm[0], cumulants_cfgs_dm[2], cumulants_cfgs_dm[4], cumulants_cfgs_dm[6])
print("dm2 k1,k3,k5,k7 =", cumulants_cfgs_dm2[0], cumulants_cfgs_dm2[2], cumulants_cfgs_dm2[4], cumulants_cfgs_dm2[6])

In [ ]:
cumulants_cfgs[1]/cumulants_cfgs_dm[1]

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
# cfgs = norm(np.load("data/field_theory/cfgs_L32_k0.4_l0.022_10k.npy", mmap_mode="r"))
cfgs_dm = np.load(
    "data/field_theory/laststep_back_samples_L32_k0.4_l0.022_nm1.npy",
    mmap_mode="r",
).astype(np.float64, copy=False)
cfgs = norm(h5py.File("data/cfgs_k=0.5_l=0.022.jld2", "r")["cfgs"])
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=100, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=100, seed=1)

In [ ]:
cfgs = norm(np.load("data/field_theory/cfgs_L32_k0.4_l0.022_10k.npy", mmap_mode="r"))
cfgs_dm = np.load(
    "data/field_theory/laststep_back_samples_L32_k0.4_l0.022_nm1.npy",
    mmap_mode="r",
).astype(np.float64, copy=False)
# cfgs = norm(h5py.File("data/cfgs_k=0.4_l=0.022.jld2", "r")["cfgs"])
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=100, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(np.concat([cfgs_dm, -cfgs_dm], axis=0), 8, n_boot=100, seed=1)

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("HMC data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
np.sqrt(0.3724/0.3960)

In [ ]:
32.0956*np.sqrt(0.5716 /0.5935)**8

In [ ]:
cfgs = norm(np.load("data/field_theory/cfgs_L32_k0.4_l0.022_10k.npy", mmap_mode="r"))


In [ ]:
from scipy import stats

In [ ]:
cfgs = norm(np.load("data/field_theory/cfgs_L32_k0.4_l0.022_10k.npy", mmap_mode="r"))
cfgs_dm = np.load("data/field_theory/laststep_back_samples_L32_k0.4_l0.022_nm1.npy", mmap_mode="r")

In [ ]:
stats.kstat(cfgs, 2,axis=0).mean()
stats.kstat(cfgs, 4,axis=0).mean()


In [ ]:
cfgs_dm = np.array(np.concatenate([samples_re], axis=0),dtype=np.float64)

In [ ]:
cfgs.shape

In [ ]:
cfgs = np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(8.4, 5),
                        subplot_kw={'xticks': [], 'yticks': []})
for i in range(3):
    for j in range(5):
        axs[i, j].imshow(cfgs[i*5+j,:,:])
plt.tight_layout(pad=0.5)
plt.show()

In [ ]:
cfgs_dm = np.load("data/cfgs_dm3_k=0.5_l=2.5_128^2_t=10-epoch=999.npy",mmap_mode="r").transpose(2,0,1)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(8.4, 5),
                        subplot_kw={'xticks': [], 'yticks': []})
for i in range(3):
    for j in range(5):
        axs[i, j].imshow(cfgs_dm[i*5+j,:,:])
plt.tight_layout(pad=0.5)
plt.show()

In [ ]:
stats.kstat(cfgs_dm, 2,axis=0).max()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs_dm, 2,axis=0),vmin=18,vmax=20.4)
plt.title("$\\kappa_2$ from DM")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs, 2,axis=0))
plt.title("$\\kappa_2$ from Data")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs, 2,axis=0),vmin=18,vmax=20.4)
plt.title("$\\kappa_2$ from Data")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs_dm, 4,axis=0))
plt.title("$\\kappa_4$ from DM")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs, 4,axis=0))
plt.title("$\\kappa_4$ from Data")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(stats.kstat(cfgs, 4,axis=0)/stats.kstat(cfgs_dm, 4,axis=0))
plt.colorbar()
plt.show()